### LangSmith hub管理提示词
---
- 使用hub加载别人分享的提示词模板
- 使用langsmith sdk来做提示词管理

In [ ]:
! pip install -qU langsmith -i https://pypi.tuna.tsinghua.edu.cn/simple

ERROR: Could not find a version that satisfies the requirement langchainhub (from versions: none)

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for langchainhub


#### 从hub上加载提示词
---
- https://smith.langchain.com 注册并获取API KEY

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

LANGCHAIN_API_KEY=os.environ.get("LANGCHAIN_API_KEY")

lsv2_<REDACTED><REDACTED>


- https://smith.langchain.com/hub 模型聚合广场

In [29]:
%pip install langchainhub
%pip install --upgrade langchainhub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langsmith import Client

import os
from dotenv import load_dotenv

load_dotenv()

langchain_key=os.getenv("langchain_key")

print(f"langchain_key: {langchain_key}")

os.environ["LANGCHAIN_API_KEY"] = langchain_key

client = Client()

# 加上 dangerously_pull_public_prompt=True
prompt = client.pull_prompt(
    "rlm/rag-prompt", 
    dangerously_pull_public_prompt=True  # <--- 关键：声明信任该公共 Prompt
)

print(prompt) 
print(prompt.input_variables)  # ['context', 'question']
print(type(prompt))

langchain_key: lsv2_<REDACTED><REDACTED>
input_variables=['context', 'question'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]
['context', 'question']
<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


#### 使用langsmith来管理提示词
---
- 创建提示词
- 测试提示词
- 迭代提示词

- 创建提示词后上传到langsmith

In [67]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv

load_dotenv()

langchain_key=os.getenv("langchain_key")

print(f"langchain_key: {langchain_key}")

os.environ["LANGCHAIN_API_KEY"] = langchain_key

client = Client()

prompt = ChatPromptTemplate(
    [
        ("system", "You are a helpful chatbot."),
        ("user", "{question}"),
    ]
)

try:
    url = client.push_prompt(
        "testprompt1", 
        object=prompt,
    )
    print(f"🎉 推送成功！你可以在 LangSmith 中查看: {url}")
except Exception as e:
    print(f"❌ 推送失败: {e}")


langchain_key: lsv2_<REDACTED><REDACTED>
❌ 推送失败: Conflict for /commits/-/testprompt1. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/commits/-/testprompt1', '{"type":"about:blank","title":"Conflict","status":409,"detail":"Nothing to commit: prompt has not changed since latest commit","error":"Nothing to commit: prompt has not changed since latest commit"}\n')


Failed to refresh cache entry test1prompt: Failed to GET /commits/-/test1prompt/latest in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/commits/-/test1prompt/latest', '{"error":"Forbidden"}\n')


* 远程拉取自己的提示词并使用千问测试

In [68]:
from langsmith import Client
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv

load_dotenv()

langchain_key=os.getenv("langchain_key")

os.environ["LANGCHAIN_API_KEY"] = langchain_key

model = ChatOpenAI(
    model="Qwen/Qwen3.8-27B",  # 模型名称          
    api_key=os.getenv("api_key"),  # 在环境变量中设置的API Key
    base_url=os.getenv("base_url"), # 在环境变量中设置的API请求地址
    temperature=0,
)

# Pull the prompt to use
# You can also specify a specific commit by passing the commit hash "my-prompt:<commit-hash>"

prompt = client.pull_prompt("testprompt1")

# Since our prompt only has one variable we could also pass in the value directly

# The code below is equivalent to formatted_prompt = prompt.format_prompt(question="What is the color of the sky?")

formatted_prompt = prompt.invoke({"question": "天空是什么颜色的？"})
print(formatted_prompt)

# Test the prompt with the LLM
response = model.invoke(formatted_prompt)
print(response)

messages=[SystemMessage(content='You are a helpful chatbot.', additional_kwargs={}, response_metadata={}), HumanMessage(content='天空是什么颜色的？', additional_kwargs={}, response_metadata={})]
content='\n\n天空的颜色会随时间和天气变化：\n\n- **白天晴天**：通常是**蓝色**\n- **日出或日落时**：可能呈现**红色、橙色、粉色或紫色**\n- **阴天**：多为**灰色**\n- **夜晚**：通常是**黑色或深蓝色**\n\n所以最常见的回答是：**天空是蓝色的**。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 64, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 42, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 64}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3.8-27B', 'system_fingerprint': '', 'id': '01a19a4366a46564801c0b39f8445100', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19a43-615b-7b02-8eee-f

* 团队迭代提示词

In [69]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv

load_dotenv()

langchain_key=os.getenv("langchain_key")

os.environ["LANGCHAIN_API_KEY"] = langchain_key

# connect to the langsmith client
client = Client()

# Define the prompt to update

new_prompt = ChatPromptTemplate(
    [
        ("system", "你是一个有趣的机器人，使用中文回复。"),
        ("user", "{question}"),
    ]
)

# Push the updated prompt making sure to use the correct prompt name

# 注意tags可以更方便进行版本管理，你也可以达成v0.0.1、v0.0.2等版本管理的效果

try:
    url = client.push_prompt(
        "testprompt1", 
        object=new_prompt,
        # tags=["v0.0.2"]
        tags=["Chinese"]
    )
    print(f"🎉 更新成功！你可以在 LangSmith 中查看: {url}")
except Exception as e:
    print(f"❌ 更新失败: {e}")

🎉 更新成功！你可以在 LangSmith 中查看: https://smith.langchain.com/prompts/testprompt1/817327ed?organizationId=bc0f0da8-3f8b-4bfa-a0c2-b9ca8042cdcf


Failed to refresh cache entry test1prompt: Failed to GET /commits/-/test1prompt/latest in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/commits/-/test1prompt/latest', '{"error":"Forbidden"}\n')
Failed to refresh cache entry test1prompt: Failed to GET /commits/-/test1prompt/latest in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/commits/-/test1prompt/latest', '{"error":"Forbidden"}\n')
Failed to refresh cache entry test1prompt: Failed to GET /commits/-/test1prompt/latest in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/commits/-/test1prompt/latest', '{"error":"Forbidden"}\n')
Failed to refresh cache entry test1prompt: Failed to GET /commits/-/test1prompt/latest in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/commits/-/test1prompt/latest', '{"error":"Forbidden"}\n')
Failed to refresh cache entr